# Stage 3: Model Training & Ensemble Classification

Notebook ini melatih **3 model classifier** untuk Auto-Severity Classification:

| Model | Teknik | Output |
|-------|--------|--------|
| **A** | IndoBERT Fine-Tuned | Semantic classifier (.pt) |
| **B** | TF-IDF + LinearSVC | N-gram classifier (.pkl) |
| **C** | TF-IDF + XGBoost | N-gram classifier (.pkl) |
| **Ensemble** | Soft Voting | Gabungan ketiga model |

**Dataset:** Teks bug report yang sudah di-preprocessing (cleaned_text)  
**Label:** Kategori severity (Mayor / Minor)

In [ ]:
# ============================================================
# Cell 1: Imports & Setup
# ============================================================
import os
import sys
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm.notebook import tqdm

# Setup paths (relatif ke root proyek)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config

# Device detection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')
print(f'Project root  : {PROJECT_ROOT}')

# Styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# ============================================================
# Cell 2: Load Data
# ============================================================
# Pilih dataset — ubah path ini sesuai kebutuhan
DATA_CSV = os.path.join(config.PROCESSED_DATA_DIR, 'Dummy_Log_Temuan_5000_1_cleaned.csv')
DATA_NPY = DATA_CSV.replace('.csv', '_embeddings.npy')

df = pd.read_csv(DATA_CSV, encoding='utf-8')
print(f'Dataset loaded: {len(df)} rows')
print(f'Columns: {list(df.columns)}')
print()

# Load pre-computed embeddings (dari Stage 2)
if os.path.exists(DATA_NPY):
    embeddings = np.load(DATA_NPY)
    print(f'Embeddings loaded: shape {embeddings.shape}')
else:
    embeddings = None
    print('WARNING: Embeddings file not found. IndoBERT akan compute ulang.')

# Pastikan tidak ada NaN
df = df.dropna(subset=[config.OUTPUT_COLUMN, config.LABEL_COLUMN]).reset_index(drop=True)
print(f'After dropna: {len(df)} rows')
df.head()

In [ ]:
# ============================================================
# Cell 3: Exploratory Data Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
label_counts = df[config.LABEL_COLUMN].value_counts()
colors = sns.color_palette('viridis', len(label_counts))
axes[0].bar(label_counts.index, label_counts.values, color=colors)
axes[0].set_title('Distribusi Label Severity', fontweight='bold')
axes[0].set_xlabel('Kategori')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Text length distribution
df['text_length'] = df[config.OUTPUT_COLUMN].str.len()
for label in df[config.LABEL_COLUMN].unique():
    subset = df[df[config.LABEL_COLUMN] == label]['text_length']
    axes[1].hist(subset, bins=30, alpha=0.6, label=label)
axes[1].set_title('Distribusi Panjang Teks per Kategori', fontweight='bold')
axes[1].set_xlabel('Panjang Teks (karakter)')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nLabel Distribution:')
print(label_counts)
print(f'\nRata-rata panjang teks: {df["text_length"].mean():.0f} karakter')

In [ ]:
# ============================================================
# Cell 4: Label Encoding & Train/Test Split
# ============================================================
# Encode labels
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[config.LABEL_COLUMN])

print('Label mapping:')
for i, cls in enumerate(le.classes_):
    count = (df['label_encoded'] == i).sum()
    print(f'  {i} = {cls} ({count} samples)')

NUM_CLASSES = len(le.classes_)
print(f'\nTotal classes: {NUM_CLASSES}')

# Stratified train/test split (80/20)
X_text = df[config.OUTPUT_COLUMN].values
y = df['label_encoded'].values

X_train_text, X_test_text, y_train, y_test, train_idx, test_idx = train_test_split(
    X_text, y, df.index.values,
    test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain set: {len(X_train_text)} samples')
print(f'Test set : {len(X_test_text)} samples')
print(f'\nTrain label distribution:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls}: {(y_train == i).sum()}')
print(f'\nTest label distribution:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls}: {(y_test == i).sum()}')

---
## Model A: IndoBERT Fine-Tuned Classifier (Semantic)

Fine-tune `indobenchmark/indobert-base-p1` dengan classification head.
IndoBERT menangani slang/singkatan secara kontekstual melalui WordPiece tokenization.

In [ ]:
# ============================================================
# Cell 5: IndoBERT — Dataset & DataLoader
# ============================================================
class SeverityDataset(Dataset):
    """PyTorch Dataset untuk IndoBERT fine-tuning."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Load tokenizer
MODEL_NAME = config.INDOBERT_MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
train_dataset = SeverityDataset(X_train_text, y_train, tokenizer, max_length=config.INDOBERT_MAX_LENGTH)
test_dataset = SeverityDataset(X_test_text, y_test, tokenizer, max_length=config.INDOBERT_MAX_LENGTH)

# Create dataloaders
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches : {len(test_loader)}')

In [ ]:
# ============================================================
# Cell 6: IndoBERT — Training
# ============================================================
# Load pre-trained model with classification head
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES
)
model_bert.to(DEVICE)

# Optimizer & Scheduler
EPOCHS = 4
LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(model_bert.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

# Training loop
train_losses = []
train_accs = []

for epoch in range(EPOCHS):
    model_bert.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in progress:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model_bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bert.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        progress.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')
    
    avg_loss = epoch_loss / len(train_loader)
    avg_acc = correct / total
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)
    print(f'  Epoch {epoch+1} — Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.4f}')

print('\nTraining complete!')

In [ ]:
# ============================================================
# Cell 7: IndoBERT — Evaluation
# ============================================================
model_bert.eval()
all_preds_bert = []
all_probs_bert = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='IndoBERT Eval'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        
        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        all_preds_bert.extend(preds.cpu().numpy())
        all_probs_bert.extend(probs.cpu().numpy())

all_preds_bert = np.array(all_preds_bert)
all_probs_bert = np.array(all_probs_bert)

acc_bert = accuracy_score(y_test, all_preds_bert)
f1_bert = f1_score(y_test, all_preds_bert, average='weighted')

print(f'\n=== Model A: IndoBERT Fine-Tuned ===')
print(f'Accuracy : {acc_bert:.4f}')
print(f'F1 Score : {f1_bert:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, all_preds_bert, target_names=le.classes_))

# Confusion Matrix
cm_bert = confusion_matrix(y_test, all_preds_bert)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — IndoBERT', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, EPOCHS+1), train_losses, 'o-', color='coral')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].plot(range(1, EPOCHS+1), train_accs, 'o-', color='teal')
axes[1].set_title('Training Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

---
## Model B: TF-IDF + LinearSVC (N-Gram Classifier)

Menggunakan fitur TF-IDF (unigram + bigram) dengan LinearSVC classifier.
Dibungkus dengan `CalibratedClassifierCV` untuk mendapatkan probabilitas (diperlukan untuk soft voting).

In [ ]:
# ============================================================
# Cell 8: TF-IDF + LinearSVC — Training & Evaluation
# ============================================================
# Build TF-IDF features
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),    # unigram + bigram
    max_features=10000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)
print(f'TF-IDF shape: {X_train_tfidf.shape}')

# Train LinearSVC (wrapped with CalibratedClassifierCV for probabilities)
base_svc = LinearSVC(class_weight='balanced', max_iter=10000, random_state=42)
model_svc = CalibratedClassifierCV(base_svc, cv=5)
model_svc.fit(X_train_tfidf, y_train)

# Predict
all_preds_svc = model_svc.predict(X_test_tfidf)
all_probs_svc = model_svc.predict_proba(X_test_tfidf)

acc_svc = accuracy_score(y_test, all_preds_svc)
f1_svc = f1_score(y_test, all_preds_svc, average='weighted')

print(f'\n=== Model B: TF-IDF + LinearSVC ===')
print(f'Accuracy : {acc_svc:.4f}')
print(f'F1 Score : {f1_svc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, all_preds_svc, target_names=le.classes_))

# Confusion Matrix
cm_svc = confusion_matrix(y_test, all_preds_svc)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svc, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — TF-IDF + LinearSVC', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## Model C: TF-IDF + XGBoost (N-Gram Classifier)

Menggunakan fitur TF-IDF yang sama (reuse) dengan XGBoost classifier.
XGBoost mendukung `predict_proba` secara native.

In [ ]:
# ============================================================
# Cell 9: TF-IDF + XGBoost — Training & Evaluation
# ============================================================
model_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(
    X_train_tfidf, y_train,
    eval_set=[(X_test_tfidf, y_test)],
    verbose=False
)

# Predict
all_preds_xgb = model_xgb.predict(X_test_tfidf)
all_probs_xgb = model_xgb.predict_proba(X_test_tfidf)

acc_xgb = accuracy_score(y_test, all_preds_xgb)
f1_xgb = f1_score(y_test, all_preds_xgb, average='weighted')

print(f'\n=== Model C: TF-IDF + XGBoost ===')
print(f'Accuracy : {acc_xgb:.4f}')
print(f'F1 Score : {f1_xgb:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, all_preds_xgb, target_names=le.classes_))

# Confusion Matrix
cm_xgb = confusion_matrix(y_test, all_preds_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — TF-IDF + XGBoost', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## Ensemble: Soft Voting

Menggabungkan probabilitas dari ketiga model:
- **IndoBERT** (semantic understanding)
- **LinearSVC** (n-gram patterns)
- **XGBoost** (n-gram patterns)

Prediksi akhir = argmax dari rata-rata probabilitas.

In [ ]:
# ============================================================
# Cell 10: Soft Voting Ensemble
# ============================================================
# Weights untuk setiap model (bisa di-tune)
W_BERT = 0.5   # IndoBERT mendapat bobot lebih tinggi (semantic)
W_SVC  = 0.25  # LinearSVC
W_XGB  = 0.25  # XGBoost

# Weighted average probabilities
ensemble_probs = (
    W_BERT * all_probs_bert +
    W_SVC  * all_probs_svc +
    W_XGB  * all_probs_xgb
)

ensemble_preds = np.argmax(ensemble_probs, axis=1)

acc_ens = accuracy_score(y_test, ensemble_preds)
f1_ens = f1_score(y_test, ensemble_preds, average='weighted')

print(f'=== Ensemble: Weighted Soft Voting ===')
print(f'Weights: IndoBERT={W_BERT}, SVC={W_SVC}, XGB={W_XGB}')
print(f'Accuracy : {acc_ens:.4f}')
print(f'F1 Score : {f1_ens:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, ensemble_preds, target_names=le.classes_))

# Confusion Matrix
cm_ens = confusion_matrix(y_test, ensemble_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Ensemble (Soft Voting)', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 11: Model Comparison
# ============================================================
results = pd.DataFrame({
    'Model': ['IndoBERT Fine-Tuned', 'TF-IDF + LinearSVC', 'TF-IDF + XGBoost', 'Ensemble (Soft Voting)'],
    'Accuracy': [acc_bert, acc_svc, acc_xgb, acc_ens],
    'F1 Score (Weighted)': [f1_bert, f1_svc, f1_xgb, f1_ens],
})
results = results.sort_values('F1 Score (Weighted)', ascending=False).reset_index(drop=True)

print('\n' + '='*60)
print('  PERBANDINGAN PERFORMA MODEL')
print('='*60)
print(results.to_string(index=False))
print('='*60)

best_model = results.iloc[0]['Model']
best_f1 = results.iloc[0]['F1 Score (Weighted)']
print(f'\nBest Model: {best_model} (F1: {best_f1:.4f})')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35
bars1 = ax.bar(x - width/2, results['Accuracy'], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, results['F1 Score (Weighted)'], width, label='F1 Score', color='coral')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Model', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Save Models

Menyimpan semua model dan artefak yang dibutuhkan untuk inference:
- IndoBERT Fine-Tuned → `.pt`
- TF-IDF Vectorizer → `.pkl`
- LinearSVC → `.pkl`
- XGBoost → `.pkl`
- Label Encoder → `.pkl`
- Ensemble Config → `.json`

In [ ]:
# ============================================================
# Cell 12: Save All Models
# ============================================================
os.makedirs(config.MODELS_DIR, exist_ok=True)

# 1. Save IndoBERT Fine-Tuned
bert_path = os.path.join(config.MODELS_DIR, 'indobert_severity.pt')
torch.save({
    'model_state_dict': model_bert.state_dict(),
    'model_name': MODEL_NAME,
    'num_classes': NUM_CLASSES,
    'max_length': config.INDOBERT_MAX_LENGTH,
    'label_classes': list(le.classes_),
    'accuracy': acc_bert,
    'f1_score': f1_bert,
}, bert_path)
print(f'[OK] IndoBERT saved: {bert_path}')

# 2. Save TF-IDF Vectorizer
tfidf_path = os.path.join(config.MODELS_DIR, 'tfidf_vectorizer.pkl')
with open(tfidf_path, 'wb') as f:
    pickle.dump(tfidf, f)
print(f'[OK] TF-IDF Vectorizer saved: {tfidf_path}')

# 3. Save LinearSVC
svc_path = os.path.join(config.MODELS_DIR, 'linearsvc_model.pkl')
with open(svc_path, 'wb') as f:
    pickle.dump(model_svc, f)
print(f'[OK] LinearSVC saved: {svc_path}')

# 4. Save XGBoost
xgb_path = os.path.join(config.MODELS_DIR, 'xgboost_model.pkl')
with open(xgb_path, 'wb') as f:
    pickle.dump(model_xgb, f)
print(f'[OK] XGBoost saved: {xgb_path}')

# 5. Save Label Encoder
le_path = os.path.join(config.MODELS_DIR, 'label_encoder.pkl')
with open(le_path, 'wb') as f:
    pickle.dump(le, f)
print(f'[OK] Label Encoder saved: {le_path}')

# 6. Save Ensemble Config
ensemble_config = {
    'weights': {
        'indobert': W_BERT,
        'linearsvc': W_SVC,
        'xgboost': W_XGB
    },
    'num_classes': NUM_CLASSES,
    'label_classes': list(le.classes_),
    'metrics': {
        'indobert': {'accuracy': float(acc_bert), 'f1': float(f1_bert)},
        'linearsvc': {'accuracy': float(acc_svc), 'f1': float(f1_svc)},
        'xgboost': {'accuracy': float(acc_xgb), 'f1': float(f1_xgb)},
        'ensemble': {'accuracy': float(acc_ens), 'f1': float(f1_ens)}
    }
}
config_path = os.path.join(config.MODELS_DIR, 'ensemble_config.json')
with open(config_path, 'w') as f:
    json.dump(ensemble_config, f, indent=2)
print(f'[OK] Ensemble Config saved: {config_path}')

print(f'\nSemua model tersimpan di: {config.MODELS_DIR}/')
for f_name in os.listdir(config.MODELS_DIR):
    f_size = os.path.getsize(os.path.join(config.MODELS_DIR, f_name))
    print(f'  {f_name:40s} {f_size/1024/1024:.1f} MB')

In [ ]:
# ============================================================
# Cell 13: Summary
# ============================================================
print('\n' + '='*60)
print('  STAGE 3 COMPLETE — MODEL TRAINING SUMMARY')
print('='*60)
print(f'  Dataset      : {os.path.basename(DATA_CSV)}')
print(f'  Total samples: {len(df)}')
print(f'  Train/Test   : {len(X_train_text)} / {len(X_test_text)}')
print(f'  Classes      : {list(le.classes_)}')
print(f'  Device       : {DEVICE}')
print(f'\n  Model Performance:')
print(results.to_string(index=False))
print(f'\n  Best Model   : {best_model}')
print(f'  Models saved : {config.MODELS_DIR}/')
print('='*60)